# PointNet++ v7.0.0 — Loss Sweep + Deployment Protocol — Train + Eval

**Tujuan v7.0.0**: Maksimalkan utilisasi dataset existing (11 subjek × 15 sesi × 10 frame/sesi) dengan:
- Multi-frame fusion di inference (mirror real deployment: 1 detik scan ≈ 10 frame)
- Loss refinement: ArcFace sweep (m=0.3/0.4/0.5), CosFace, Sub-Center ArcFace K=3, combined (Hybrid)
- Protokol evaluasi lebih realistis: cross-session pair, open-set LOSO

**Notebook ini**: Training + single-frame evaluation semua varian loss.
**Lanjutkan di**: `v7_multiframe_compare.ipynb` untuk multi-frame fusion ablation + open-set LOSO.

**Perubahan v7 vs v6**:
- Subjek: 11 (gede diaktifkan, v6 = 10)
- Frame per sesi: tetap 1 median frame untuk training
- Loss variants: standard (triplet), arcface m=0.3, arcface m=0.4, arcface m=0.5 (v6 default),
  cosface, subcenter_arcface (K=3), hybrid (arcface+triplet)
- `--frame-sampling median` default; `--frame-sampling random` tersedia untuk ablation C1
- Tidak ada klaim p < 0.05 sebagai primary; primary metric: effect size Cohen's d + bootstrap CI


## 1. Setup — GitHub Clone + Dataset in Repo

In [ ]:
from google.colab import userdata
import os, sys, subprocess
from pathlib import Path

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/RZulfikri/Thesis.git'
REPO_DIR     = Path('/content/Thesis')
PROJECT_ROOT = REPO_DIR / '3DCNN'
COLAB_BRANCH = 'colab'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin

os.chdir(str(REPO_DIR))
ret = subprocess.run(['git', 'checkout', COLAB_BRANCH], capture_output=True)
if ret.returncode != 0:
    !git checkout -b {COLAB_BRANCH}
else:
    !git merge origin/main --no-edit --strategy-option theirs 2>/dev/null || true

!git config user.email 'colab-runner@thesis.local'
!git config user.name  'Colab Runner'

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))

# Verifikasi dataset & subjek v7
from utils.dataset_lowdata import scan_sessions, DROPPED_SUBJECTS
sessions = scan_sessions('dataset')
print(f'Subjek aktif v7: {sorted(sessions.keys())} ({len(sessions)} subjek)')
print(f'DROPPED_SUBJECTS: {DROPPED_SUBJECTS}')
print(f'Total sesi: {sum(len(v) for v in sessions.values())}')
print('Setup OK')

## 2. TensorBoard

In [ ]:
import os
os.environ['TENSORBOARD_BINARY'] = '/usr/local/bin/tensorboard'
%load_ext tensorboard
%tensorboard --logdir /content/Thesis/3DCNN/runs/v7_lowdata

## 3. Konfigurasi + GPU Probe

In [ ]:
import subprocess, os, sys, gc
import torch

SEEDS = [42, 123, 2024, 7, 31337, 0, 1, 2, 3, 4]

# ============================================================
# Varian loss v7.0.0:
#   standard         : PointNet++ + Triplet (baseline)
#   arcface_m03      : ArcFace m=0.3, s=30
#   arcface_m04      : ArcFace m=0.4, s=30
#   arcface_m05      : ArcFace m=0.5, s=30 (sama dengan v6 default)
#   arcface_s64      : ArcFace m=0.5, s=64
#   cosface          : CosFace m=0.35, s=30
#   subcenter        : Sub-Center ArcFace K=3, m=0.5, s=30
#   hybrid           : ArcFace + Triplet (alpha=0.7)
# ============================================================
VARIANTS = [
    # (variant_name, loss_type, arc_margin, arc_scale, subcenter_k)
    ('standard',    'triplet',          0.3,  30.0, 3),
    ('arcface_m03', 'arcface',          0.3,  30.0, 3),
    ('arcface_m04', 'arcface',          0.4,  30.0, 3),
    ('arcface_m05', 'arcface',          0.5,  30.0, 3),
    ('arcface_s64', 'arcface',          0.5,  64.0, 3),
    ('cosface',     'cosface',          0.35, 30.0, 3),
    ('subcenter',   'subcenter_arcface',0.5,  30.0, 3),
    ('hybrid',      'hybrid',           0.5,  30.0, 3),
]

DATA_DIR = PROJECT_ROOT / 'dataset'
RUNS_DIR = PROJECT_ROOT / 'runs'   / 'v7_lowdata'
EVAL_DIR = PROJECT_ROOT / 'eval_results' / 'v7_lowdata'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS          = 120
FINETUNE_EPOCHS = 30

N_POINTS             = 16384
TARGET_VRAM_FRACTION = 0.75
MAX_BS_FOR_LARGE_N   = 192
AMP_MODE             = 'bf16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'fp16'
PRELOAD_AUGMENT      = True
NUM_WORKERS          = 8

try:
    out = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=memory.total,name', '--format=csv,noheader,nounits'], text=True
    )
    line = out.strip().split('\n')[0]
    vram_str, gpu_name = [x.strip() for x in line.split(',')]
    VRAM_GB = int(vram_str) / 1024
    GPU_NAME = gpu_name
except Exception:
    VRAM_GB = 0; GPU_NAME = 'Unknown'

target_vram_gb = VRAM_GB * TARGET_VRAM_FRACTION
print(f'GPU: {GPU_NAME} ({VRAM_GB:.1f} GB) | target={target_vram_gb:.1f} GB | AMP={AMP_MODE}')

def probe_max_batch_size(n_points, target_gb, amp_dtype):
    sys.path.insert(0, str(PROJECT_ROOT))
    from models.siamese import SiamesePalmNet
    device = torch.device('cuda')
    def _try_bs(bs):
        torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
        try:
            model = SiamesePalmNet(geom_dim=13, use_geom=False, use_aux_loss=False,
                                   n_subjects=11, siamese_mode='concat').to(device)
            model.train()
            opt = torch.optim.Adam(model.parameters(), lr=1e-3, fused=True)
            pts  = torch.randn(bs, n_points, 6, device=device)
            geom = torch.randn(bs, 13, device=device)
            ctx  = torch.amp.autocast('cuda', dtype=amp_dtype) if amp_dtype else torch.amp.autocast('cuda', enabled=False)
            with ctx:
                emb = model.encode(pts, geom); loss = emb.sum()
            loss.backward(); opt.step(); torch.cuda.synchronize()
            peak = torch.cuda.max_memory_allocated() / (1024**3)
            del model, opt, pts, geom, emb, loss; torch.cuda.empty_cache(); gc.collect()
            return peak
        except (torch.cuda.OutOfMemoryError, RuntimeError):
            torch.cuda.empty_cache(); gc.collect(); return float('inf')
    amp_dt = torch.bfloat16 if AMP_MODE == 'bf16' else (torch.float16 if AMP_MODE == 'fp16' else None)
    bs_low, best_bs, best_peak = 32, 32, 0.0
    bs = 32
    while True:
        peak = _try_bs(bs)
        if peak == float('inf'):
            bs_high = bs; break
        best_bs, best_peak = bs, peak
        if peak >= target_gb * 0.95:
            bs_high = int(bs * (target_gb / peak) * 1.1); break
        bs_low = bs; bs *= 2
    else:
        bs_high = bs_low * 2
    while bs_high - bs_low > 16:
        m = (bs_low + bs_high) // 2
        peak = _try_bs(m)
        if peak == float('inf') or peak > VRAM_GB * 0.95:
            bs_high = m
        else:
            if peak <= target_gb:
                best_bs, best_peak = m, peak; bs_low = m
            else:
                bs_high = m
    final_bs = max(32, int(best_bs * 0.9))
    final_bs = min(final_bs, MAX_BS_FOR_LARGE_N)
    print(f'  → BS={final_bs}, peak~{best_peak:.1f}GB')
    return final_bs, best_peak

amp_dt = torch.bfloat16 if AMP_MODE == 'bf16' else (torch.float16 if AMP_MODE == 'fp16' else None)
BS, peak_gb = probe_max_batch_size(N_POINTS, target_vram_gb, amp_dt)

# v7: 11 subjek × 8 sesi = 88 training frames
REPEAT = max(4, -(-min(BS, 512) * 4 // 88))

print('\n' + '='*60)
print(f'AUTO-TUNE: BS={BS}  REPEAT={REPEAT}  N_POINTS={N_POINTS}  AMP={AMP_MODE}')
print(f'Training frames: 88 (11 subj × 8 sesi)  dataset_len={88*REPEAT}')
print(f'Variants: {[v[0] for v in VARIANTS]}')
print('='*60)
torch.cuda.empty_cache(); gc.collect()

## 4. Cleanup (Opsional)

In [ ]:
import shutil
RUN_CLEANUP = False
if RUN_CLEANUP:
    for d in [RUNS_DIR, EVAL_DIR]:
        if d.exists(): shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)
    print('Cleanup selesai.')
else:
    print('Skipped (RUN_CLEANUP = False)')

## 5. Helper: Training + Eval

In [ ]:
def run_training(variant_name, loss_type, arc_margin, arc_scale, subcenter_k, seed):
    out_dir = RUNS_DIR / variant_name / f'seed_{seed}'
    ckpt    = out_dir / 'best.pth'
    if ckpt.exists():
        print(f'  SKIP training {variant_name} seed={seed} (done)')
        return True

    amp_flag     = f'--amp {AMP_MODE}' if AMP_MODE != 'none' else ''
    preload_flag = '--preload-augment' if PRELOAD_AUGMENT else ''

    # Argumen khusus per loss type
    loss_args = f'--loss {loss_type} --arcface-margin {arc_margin} --arcface-scale {arc_scale}'
    if loss_type == 'subcenter_arcface':
        loss_args += f' --subcenter-k {subcenter_k}'

    cmd = (
        f'python {PROJECT_ROOT / "train.py"} '
        f'--data_dir {DATA_DIR} '
        f'--output_dir {out_dir} '
        f'--seed {seed} '
        f'--fixed_split '
        f'--frames-per-session 1 '
        f'{loss_args} '
        f'--val-metric pair_eer --no-early-stop '
        f'--epochs {EPOCHS} --finetune_epochs {FINETUNE_EPOCHS} '
        f'--batch_size {BS} --n_points {N_POINTS} '
        f'--num_workers {NUM_WORKERS} '
        f'--repeat {REPEAT} '
        f'{amp_flag} {preload_flag} '
        f'--siamese-mode concat'
    )
    print(f'\n{"="*60}')
    print(f'TRAIN: {variant_name} | seed={seed} | loss={loss_type} m={arc_margin} s={arc_scale}')
    print(f'{"="*60}')
    !{cmd}
    return ckpt.exists()


def run_eval(variant_name, seed, split):
    ckpt       = RUNS_DIR / variant_name / f'seed_{seed}' / 'best.pth'
    eval_out   = EVAL_DIR / variant_name / f'seed_{seed}' / split
    result_f   = eval_out / 'results.json'

    if not ckpt.exists():
        print(f'  SKIP eval {variant_name} seed={seed} split={split} (checkpoint missing)')
        return False
    if result_f.exists():
        print(f'  SKIP eval {variant_name} seed={seed} split={split} (done)')
        return True

    eval_out.mkdir(parents=True, exist_ok=True)
    cmd = (
        f'python {PROJECT_ROOT / "evaluate.py"} '
        f'--data_dir {DATA_DIR} '
        f'--frames-per-session 1 '
        f'--eval-split {split} '
        f'--checkpoints "{variant_name}={ckpt}" '
        f'--output_dir {eval_out} '
        f'--save_scores'
    )
    print(f'  EVAL: {variant_name} seed={seed} {split}')
    !{cmd}
    return result_f.exists()

print('Helpers ready (v7)')

## 6. Runtime Shutdown Guard + Git Save

In [ ]:
import atexit, subprocess
_shutdown_triggered = False

def shutdown_colab(silent=False):
    global _shutdown_triggered
    if _shutdown_triggered: return
    _shutdown_triggered = True
    if not silent:
        print('Shutting down Colab runtime...')
    try:
        from google.colab import runtime; runtime.unassign()
    except Exception as e:
        if not silent: print(f'Shutdown error: {e}')

atexit.register(shutdown_colab, silent=True)

def git_save(message, push=False):
    """Commit & push semua file (tanpa filter ukuran)."""
    os.chdir(str(REPO_DIR))
    subprocess.run(['git', 'add', '-A'], cwd=str(REPO_DIR))
    if subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=str(REPO_DIR)).returncode == 0:
        print('  (nothing to commit)'); os.chdir(str(PROJECT_ROOT)); return
    ret = subprocess.run(['git', 'commit', '-m', message], cwd=str(REPO_DIR),
                         capture_output=True, text=True)
    if ret.returncode == 0:
        print(f'Committed: {message}')
    else:
        print(f'Commit error: {ret.stderr[:300]}')
    if push:
        ret = subprocess.run(['git', 'push', 'origin', COLAB_BRANCH],
                             cwd=str(REPO_DIR), capture_output=True, text=True)
        if ret.returncode == 0:
            print('Pushed OK')
        else:
            print(f'Push error: {ret.stderr[:300]}')
    os.chdir(str(PROJECT_ROOT))

print('Shutdown guard + git_save ready')

## 7. Training — Semua Varian × 10 Seeds

Total: 8 varian × 10 seeds = 80 training runs. Untuk menjalankan varian tertentu saja,
edit `VARIANTS` di cell konfigurasi di atas.

In [ ]:
for variant_name, loss_type, arc_margin, arc_scale, subcenter_k in VARIANTS:
    print(f'\n### Variant: {variant_name} ###')
    for seed in SEEDS:
        run_training(variant_name, loss_type, arc_margin, arc_scale, subcenter_k, seed)
    # Incremental save setelah tiap varian selesai (safety: Colab bisa timeout)
    git_save(f'v7.1.0: training {variant_name} complete (10 seeds)', push=True)

## 8. Evaluasi — Test + Holdout (single-frame, cross-session bawaan split)

In [ ]:
for variant_name, *_ in VARIANTS:
    for seed in SEEDS:
        run_eval(variant_name, seed, 'test')
        run_eval(variant_name, seed, 'holdout')
    # Incremental save setelah eval tiap varian
    git_save(f'v7.1.0: eval {variant_name} complete', push=True)

In [ ]:
# Simpan ke git setelah semua selesai
git_save('v7.0.0: all variants training + eval complete', push=True)

## 9. Quick Summary — Single-Frame EER

In [ ]:
import json, numpy as np

print(f'{"Variant":<18} {"Test EER (mean±std)":<24} {"Holdout EER (mean±std)":<24}')
print('-' * 70)

for variant_name, *_ in VARIANTS:
    test_eers, hold_eers = [], []
    for seed in SEEDS:
        for split, lst in [('test', test_eers), ('holdout', hold_eers)]:
            f = EVAL_DIR / variant_name / f'seed_{seed}' / split / 'results.json'
            if f.exists():
                with open(f) as fp:
                    r = json.load(fp)
                lst.append(r[0]['eer'])
    t_str = f'{np.mean(test_eers):.4f}±{np.std(test_eers):.4f}' if test_eers else 'missing'
    h_str = f'{np.mean(hold_eers):.4f}±{np.std(hold_eers):.4f}' if hold_eers else 'missing'
    print(f'{variant_name:<18} {t_str:<24} {h_str:<24}')

print('\nLanjut ke v7_multiframe_compare.ipynb untuk multi-frame fusion + LOSO analysis')

## 10. t-SNE Embedding Visualization

Visualisasi distribusi embedding di ruang 2D untuk setiap varian (seed=42).
Warna = subjek identity. Cluster yang terpisah = model bisa membedakan subjek.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from pathlib import Path

sys.path.insert(0, str(PROJECT_ROOT))
from models.siamese import SiamesePalmNet
from utils.dataset_lowdata import build_lowdata_splits
from utils.normalizer import GeometryNormalizer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TSNE_SEED = 42
N_PTS_TSNE = 8192

def load_model_for_viz(variant, seed):
    ckpt_path = RUNS_DIR / variant / f'seed_{seed}' / 'best.pth'
    if not ckpt_path.exists():
        return None
    model = SiamesePalmNet(
        geom_dim=13, use_geom=False, use_aux_loss=False,
        n_subjects=11, siamese_mode='concat',
    ).to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    state = ckpt.get('model_state_dict', ckpt)
    if any(k.startswith('encoder.proj.') for k in state.keys()):
        state = {k.replace('encoder.proj.', 'encoder.proj_no_geom.'): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    model.eval()
    return model

def extract_embeddings(model, splits, split_names=('test', 'holdout')):
    """Encode semua frame dari split tertentu, return embeddings + labels."""
    all_embs, all_labels, all_splits = [], [], []
    geom_dummy = torch.zeros(1, 13, device=DEVICE)
    for sp in split_names:
        for label, frames in splits[sp].items():
            for frame_path in frames:
                cloud = np.load(Path(frame_path) / 'cnn_input.npy').astype(np.float32)
                idx = np.random.default_rng(42).choice(len(cloud), min(N_PTS_TSNE, len(cloud)), replace=False)
                pts = torch.from_numpy(cloud[idx]).unsqueeze(0).to(DEVICE)
                with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
                    emb = model.encode(pts, geom_dummy)
                all_embs.append(emb.cpu().numpy().squeeze())
                all_labels.append(label)
                all_splits.append(sp)
    return np.array(all_embs), all_labels, all_splits

# Build splits
splits = build_lowdata_splits(str(DATA_DIR))
label_names = sorted(set(splits['test'].keys()))
label_to_idx = {l: i for i, l in enumerate(label_names)}

# Plot t-SNE per varian (seed=42)
n_v = len(VARIANTS)
ncols = min(4, n_v)
nrows = (n_v + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
axes = np.array(axes).flatten()

cmap = plt.cm.get_cmap('tab20', len(label_names))

for i, (variant_name, *_) in enumerate(VARIANTS):
    ax = axes[i]
    model = load_model_for_viz(variant_name, TSNE_SEED)
    if model is None:
        ax.set_title(f'{variant_name} — no checkpoint'); continue

    embs, labels, split_tags = extract_embeddings(model, splits)
    tsne = TSNE(n_components=2, perplexity=min(30, len(embs)-1), random_state=42)
    embs_2d = tsne.fit_transform(embs)
    label_idx = [label_to_idx[l] for l in labels]

    scatter = ax.scatter(embs_2d[:, 0], embs_2d[:, 1], c=label_idx, cmap=cmap,
                         s=15, alpha=0.7, edgecolors='none')
    ax.set_title(f'{variant_name}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    del model; torch.cuda.empty_cache()

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=cmap(label_to_idx[l]),
           markersize=6, label=l) for l in label_names]
fig.legend(handles=handles, loc='lower center', ncol=min(6, len(label_names)),
           fontsize=8, bbox_to_anchor=(0.5, -0.02))
plt.suptitle('t-SNE Embedding per Variant (seed=42, test+holdout)', y=1.01)
plt.tight_layout()
plt.savefig(RUNS_DIR / 'tsne_all_variants.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Disimpan: {RUNS_DIR}/tsne_all_variants.png')

## 11. Confusion Matrix (Single-Frame, seed=42)

Confusion matrix berdasarkan nearest-neighbor matching di embedding space.
Setiap frame test di-match ke frame train terdekat (cosine similarity).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def compute_confusion_matrix(model, splits, label_names):
    """Nearest-neighbor classification: encode train+test, match test ke train terdekat."""
    geom_dummy = torch.zeros(1, 13, device=DEVICE)

    def encode_split(split_frames):
        embs, labs = [], []
        for label, frames in split_frames.items():
            for fp in frames:
                cloud = np.load(Path(fp) / 'cnn_input.npy').astype(np.float32)
                idx = np.random.default_rng(42).choice(len(cloud), min(N_PTS_TSNE, len(cloud)), replace=False)
                pts = torch.from_numpy(cloud[idx]).unsqueeze(0).to(DEVICE)
                with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.float16):
                    emb = model.encode(pts, geom_dummy)
                embs.append(emb.cpu().numpy().squeeze())
                labs.append(label)
        return np.array(embs), labs

    train_embs, train_labs = encode_split(splits['train'])
    test_embs, test_labs = encode_split(splits['test'])

    # Cosine similarity → nearest neighbor
    train_norm = train_embs / (np.linalg.norm(train_embs, axis=1, keepdims=True) + 1e-9)
    test_norm = test_embs / (np.linalg.norm(test_embs, axis=1, keepdims=True) + 1e-9)
    sim = test_norm @ train_norm.T
    pred_idx = sim.argmax(axis=1)
    preds = [train_labs[i] for i in pred_idx]

    cm = confusion_matrix(test_labs, preds, labels=label_names)
    return cm, test_labs, preds

# Plot confusion matrix per varian
n_v = len(VARIANTS)
ncols = min(4, n_v)
nrows = (n_v + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
axes = np.array(axes).flatten()

for i, (variant_name, *_) in enumerate(VARIANTS):
    ax = axes[i]
    model = load_model_for_viz(variant_name, TSNE_SEED)
    if model is None:
        ax.set_title(f'{variant_name} — no checkpoint'); continue

    cm, true_labs, pred_labs = compute_confusion_matrix(model, splits, label_names)
    acc = np.trace(cm) / cm.sum() * 100

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False, xticks_rotation=45)
    ax.set_title(f'{variant_name} (acc={acc:.1f}%)', fontsize=10)
    ax.set_xlabel(''); ax.set_ylabel('')
    del model; torch.cuda.empty_cache()

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrix — Nearest-Neighbor (seed=42, test split)', y=1.01)
plt.tight_layout()
plt.savefig(RUNS_DIR / 'confusion_matrix_all_variants.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'Disimpan: {RUNS_DIR}/confusion_matrix_all_variants.png')

## Auto-Shutdown

In [ ]:
import time
SHUTDOWN_DELAY = 60
print(f'ALL DONE — v7.0.0 train+eval selesai! Shutdown dalam {SHUTDOWN_DELAY}s...')
try:
    for r in range(SHUTDOWN_DELAY, 0, -10):
        print(f'  {r}s...'); time.sleep(min(10, r))
    shutdown_colab()
except KeyboardInterrupt:
    print('Shutdown dibatalkan.'); _shutdown_triggered = False